# Module 3 — Local Model Selection and Benchmarking

Companion notebook to [`docs/modules/03_local_model_selection_and_benchmarking.md`](../docs/modules/03_local_model_selection_and_benchmarking.md).

This notebook demonstrates the benchmark harness (`scripts/module_03/`) two ways:

1. **Against a fake, fast, deterministic model** — proves the harness (dataset loading,
   prompt building, scoring, aggregation, scorecard rendering) actually works, with no
   installs needed.
2. **Against real Ollama models, if available** — on this machine that's expected to skip
   (see the machine constraint in `PROGRESS.md`); on a resourced Mac it produces a real
   3-model comparison.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "scripts" / "module_01"))
sys.path.insert(0, str(REPO_ROOT / "scripts" / "module_03"))
print(f"Repo root: {REPO_ROOT}")

Repo root: /Users/bhakti/workspace/local_ai_app


## 1. Load the golden sets

Six task-type datasets under `evals/golden_sets/`, each a frozen, versioned regression set
(theory doc §10).

In [2]:
import run_benchmark as bench

for name in bench.TASK_FILES:
    path = bench.GOLDEN_SETS_DIR / name
    dataset = bench.load_dataset(path)
    print(f"{name:22s} {len(dataset)} records")

summarization.jsonl    6 records
extraction.jsonl       6 records
classification.jsonl   8 records
code.jsonl             6 records
rag.jsonl              5 records
tool_calling.jsonl     5 records


## 2. Run the harness against a fake model (proves it works, no installs needed)

A tiny rule-based fake model plays back a plausible-but-imperfect answer for each task type,
so we see both correct and incorrect scoring in the output — not just a wall of 1.0s.

In [3]:
import random

def fake_model(model: str, prompt: str) -> str:
    """A deterministic stand-in model: gets classification/rag right, is intentionally
    imperfect at extraction (to show the scorer catching a missing field)."""
    rng = random.Random(hash((model, prompt)) % (2**31))
    if "Respond with only the label" in prompt:
        text = prompt.split("Text:")[-1].lower()
        if "charg" in text:
            return "billing"
        if "deliver" in text or "ship" in text or "address" in text:
            return "shipping"
        if "crash" in text or "login" in text or "blank" in text:
            return "technical_support"
        return "general_inquiry"
    if "Extract the following fields" in prompt:
        # Deliberately "forgets" one field sometimes, like a real small model would.
        return '{"name": "Maria"}' if rng.random() < 0.3 else '{"name": "Maria", "age": 29, "city": "Austin"}'
    if "Summarize the following text" in prompt:
        return "This is a generic summary that misses most of the specific facts."
    if "Answer only using the provided context" in prompt:
        # crude: if the question looks unanswerable from context, refuse; else echo a citation
        return "I don't know based on the provided documents." if "CEO" in prompt or "parking" in prompt else "See [doc1] for details."
    if "You can call these functions" in prompt:
        return "NO_TOOL"
    return "a plausible but generic response"

dataset_paths = {name: bench.GOLDEN_SETS_DIR / name for name in bench.TASK_FILES}
results = bench.run_full_benchmark(["fake-model-a", "fake-model-b"], dataset_paths, fake_model)

In [4]:
from IPython.display import Markdown, display

display(Markdown(bench.comparison_table(results)))

| Model | summarization | extraction | classification | code | rag | tool_calling |
|---|---:|---:|---:|---:|---:|---:|
| fake-model-a | 0.00 | 0.33 | 1.00 | 0.00 | 1.00 | 0.20 |
| fake-model-b | 0.00 | 0.17 | 1.00 | 0.00 | 1.00 | 0.20 |

The extraction and tool-calling scores below 1.0 are expected and correct: the fake model
sometimes drops the `age`/`city` fields, and always declines to call a tool (which is the
*right* answer for the negative-case record but the *wrong* answer for the four positive
records) — proving the scorers actually discriminate rather than rubber-stamping everything.

In [5]:
display(Markdown(bench.scorecard_quality_table(results["fake-model-a"])))

| Task | Score | Records |
|---|---:|---:|
| summarization | 0.00 | 6 |
| extraction | 0.33 | 6 |
| classification | 1.00 | 8 |
| code | 0.00 | 6 |
| rag | 1.00 | 5 |
| tool_calling | 0.20 | 5 |

## 3. Run against real models, if Ollama is available

On the machine this course is authored on, this is expected to print a skip notice — no
model runtime is installed here by design (see `PROGRESS.md`). On a resourced Mac with
Ollama running and models pulled, this produces a real scorecard comparison.

In [6]:
from ollama_probe import is_ollama_available, list_local_models

if is_ollama_available():
    available = set(list_local_models())
    candidates = [m for m in ["qwen2.5:1.5b", "qwen2.5:3b", "qwen2.5:7b"] if m in available]
    if len(candidates) >= 3:
        real_results = bench.run_full_benchmark(candidates, dataset_paths, bench.default_generate_fn)
        display(Markdown(bench.comparison_table(real_results)))
    else:
        print(f"Ollama is running but fewer than 3 candidate models are pulled ({candidates}).")
        print("Run: ollama pull qwen2.5:1.5b && ollama pull qwen2.5:3b && ollama pull qwen2.5:7b")
else:
    print("SKIPPED: Ollama is not reachable at http://localhost:11434.")
    print("This machine deliberately has no local model runtime installed (see PROGRESS.md).")
    print("On a resourced Mac: pull >=3 models, then re-run this cell for a real comparison.")

SKIPPED: Ollama is not reachable at http://localhost:11434.
This machine deliberately has no local model runtime installed (see PROGRESS.md).
On a resourced Mac: pull >=3 models, then re-run this cell for a real comparison.


## 4. Take it to the command line

```bash
uv run python scripts/module_03/run_benchmark.py --models qwen2.5:1.5b qwen2.5:3b qwen2.5:7b \
    > reports/module_03_benchmark_raw_output.md
```

Then fold the real comparison table into
[`reports/module_03_local_model_selection_report.md`](../reports/module_03_local_model_selection_report.md),
and fill one copy of
[`reports/model_scorecard_TEMPLATE.md`](../reports/model_scorecard_TEMPLATE.md) per model.